# Backend gratuito del Generador de Actas
Este notebook inicia el servidor FastAPI en Google Colab y crea una URL pública temporal con Cloudflare Tunnel.

**Antes de ejecutar:** sube la carpeta `backend` del repositorio a Google Drive o clona tu repositorio de GitHub en la siguiente celda.

In [ ]:
# OPCIÓN A: clonar tu repositorio público de GitHub
REPO_URL = 'PEGA_AQUI_LA_URL_DE_TU_REPOSITORIO'
!rm -rf /content/generador-actas
!git clone $REPO_URL /content/generador-actas
%cd /content/generador-actas/backend

In [ ]:
!pip -q install -r requirements.txt
!apt-get -qq update
!apt-get -qq install -y ffmpeg

## Iniciar API y túnel
La celda descarga `cloudflared`, arranca la API y muestra una URL `https://....trycloudflare.com`. Pégala en la página web, en **Servidor de procesamiento**.

In [ ]:
import os, re, time, subprocess, textwrap, requests

# Iniciar FastAPI
api = subprocess.Popen(['uvicorn','app:app','--host','0.0.0.0','--port','8000'])
time.sleep(3)

# Instalar cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# Crear túnel temporal
tunnel = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://localhost:8000','--no-autoupdate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
for _ in range(80):
    line = tunnel.stdout.readline()
    print(line, end='')
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break
if public_url:
    print('\n\n✅ URL DEL BACKEND:', public_url)
else:
    print('No se detectó la URL. Revisa la salida anterior.')

### Mantener Colab activo
Mientras esta sesión esté conectada, la página podrá usar el backend. Si Colab se desconecta, vuelve a ejecutar el notebook y actualiza la URL en la página.